# SURE-Research Pipeline

**Single prerequisite:** `candidate_posts_eng10.jsonl` in your Google Drive `SURE-Research/` folder.

**How to run:**
1. Edit **Cell 1 (CONFIG)** — paste your API key, adjust topics/axes if needed
2. Set runtime to **T4 GPU** (`Runtime → Change runtime type → T4 GPU`)
3. `Runtime → Run all` — walk away

All intermediate files are saved to Google Drive. If the session drops, re-run and it picks up where it left off.


In [ ]:
# ── CONFIG — edit this cell only ─────────────────────────────────────────────

ANTHROPIC_API_KEY = "sk-ant-..."       # your Anthropic API key
PIPELINE          = "both"             # "dataset", "sae", or "both"
DRIVE_FOLDER      = "SURE-Research"    # folder name in Google Drive root

TOP_N_PER_TOPIC   = 150                # posts per topic in final dataset
N_PER_TOPIC_SAE   = 41                 # posts per topic selected for SAE rewrites

# Topics: cluster_id comes from topic_info.csv after DS2 runs
TOPICS = [
    {"cluster_id": 0,   "name": "Pets"},
    {"cluster_id": 10,  "name": "Music"},
    {"cluster_id": 11,  "name": "Books & Reading"},
    {"cluster_id": 12,  "name": "Climate & Energy"},
    {"cluster_id": 15,  "name": "Sports"},
    {"cluster_id": 20,  "name": "Birds & Nature"},
    {"cluster_id": 23,  "name": "Food & Cooking"},
    {"cluster_id": 33,  "name": "Gardening & Plants"},
    {"cluster_id": 39,  "name": "Movies & Film"},
    {"cluster_id": 48,  "name": "Video Games"},
    {"cluster_id": 49,  "name": "Space & Astronomy"},
    {"cluster_id": 51,  "name": "Ancient History"},
    {"cluster_id": 52,  "name": "Fitness & Gym"},
    {"cluster_id": 63,  "name": "Politics"},
    {"cluster_id": 78,  "name": "Higher Education"},
    {"cluster_id": 83,  "name": "Economy & Jobs"},
    {"cluster_id": 106, "name": "Capitalism"},
]

AXES = [
    {
        "name": "reading_level", "short": "read_lvl", "color": "#6c8ebf",
        "label_description": "0=simple vocabulary/short sentences, 1=academic/complex",
        "rewrite_up":   "more specialist vocabulary and complex sentence structure",
        "rewrite_down": "simpler vocabulary and shorter sentences, as if explaining to a general audience",
    },
    {
        "name": "background", "short": "backgrnd", "color": "#82b366",
        "label_description": "0=assumes no prior knowledge, 1=assumes expert knowledge",
        "rewrite_up":   "assume more prior knowledge — skip context and get straight to the point",
        "rewrite_down": "unpack more background context so a newcomer can follow along",
    },
    {
        "name": "abstract_concrete", "short": "abst_conc", "color": "#d6b656",
        "label_description": "0=vague/general claims, 1=specific facts/examples/numbers",
        "rewrite_up":   "more concrete — replace vague claims with specific named facts, events, or people",
        "rewrite_down": "more abstract — remove specific examples and speak in general terms",
    },
    {
        "name": "tone", "short": "tone", "color": "#ae4132",
        "label_description": "0=analytical/neutral, 1=emotional/charged",
        "rewrite_up":   "more emotionally charged and urgent",
        "rewrite_down": "more measured and analytical — cool the emotional temperature down",
    },
    {
        "name": "humor", "short": "humor", "color": "#9673a6",
        "label_description": "0=earnest, 1=very humorous",
        "rewrite_up":   "wittier and more playful — add dry humor or sarcasm",
        "rewrite_down": "more earnest and serious — remove any wit or comedy",
    },
    {
        "name": "narrativity", "short": "narrat", "color": "#d79b00",
        "label_description": "0=pure argument/facts, 1=story/anecdote",
        "rewrite_up":   "more story-like — add a brief personal anecdote or scene-setting opener",
        "rewrite_down": "pure assertion style — remove personal framing, just make the argument directly",
    },
    {
        "name": "grounding", "short": "ground", "color": "#23748a",
        "label_description": "0=direct statement, 1=analogy/example-driven",
        "rewrite_up":   "more example-driven — add a concrete analogy or real-world comparison",
        "rewrite_down": "more general — remove analogies and examples, state the point directly",
    },
]


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import subprocess, os, sys, json, yaml

print("Installing packages...")
subprocess.run([
    "pip", "install", "-q",
    "anthropic", "scikit-learn", "pandas", "pyyaml",
    "transformers", "accelerate", "sentence-transformers",
    "bertopic", "umap-learn", "hdbscan",
], check=True)

from google.colab import drive
drive.mount("/content/drive")

WORK_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {WORK_DIR}")

# Clone repo for config.py and sae4_labeler.py
REPO_DIR = "/content/sure-research"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "https://github.com/Jiahao-Jerry/SURE-research", REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"])
sys.path.insert(0, REPO_DIR)

# Write pipeline_config.yaml
with open("pipeline_config.yaml", "w") as f:
    yaml.dump({"topics": TOPICS, "axes": AXES}, f, allow_unicode=True)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

AXIS_NAMES = [a["name"] for a in AXES]
OUTPUT_DS  = f"{len(TOPICS) * TOP_N_PER_TOPIC}_posts.jsonl"

# Verify raw data exists (accept .gz or unzipped)
if os.path.exists("candidate_posts_eng10.jsonl.gz"):
    INPUT_FILE = "candidate_posts_eng10.jsonl.gz"
elif os.path.exists("candidate_posts_eng10.jsonl"):
    INPUT_FILE = "candidate_posts_eng10.jsonl"
else:
    raise FileNotFoundError(
        "candidate_posts_eng10.jsonl(.gz) not found in Google Drive SURE-Research/ folder. "
        "Upload it first then re-run."
    )
print(f"  Raw data : {INPUT_FILE}")

print(f"\nConfig:")
print(f"  Pipeline : {PIPELINE}")
print(f"  Topics   : {len(TOPICS)}")
print(f"  Axes     : {AXIS_NAMES}")
print(f"  Output   : {OUTPUT_DS}")
print("\n✓ Setup complete")


---
## DS1 — BGE-M3 Embeddings
~2h on T4 GPU. Skipped if `embeddings.npy` already in Drive.


In [ ]:
# ── DS1: BGE-M3 Embeddings (GPU) ─────────────────────────────────────────────
print(f"DS1 checking in: {os.getcwd()}")
print(f"  embeddings.npy : {os.path.exists('embeddings.npy')}")
print(f"  post_ids.npy   : {os.path.exists('post_ids.npy')}")
if os.path.exists("embeddings.npy") and os.path.exists("post_ids.npy"):
    import numpy as np
    shape = np.load("embeddings.npy", mmap_mode="r").shape
    print(f"✓ Skipping BGE-M3 embedding — embeddings.npy exists {shape}")
else:
    import json as _json, numpy as np, torch
    from sentence_transformers import SentenceTransformer

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    MODEL_NAME = "BAAI/bge-m3"
    BATCH_SIZE = 512

    print(f"Loading model {MODEL_NAME}...")
    model = SentenceTransformer(MODEL_NAME, device="cuda",
                                model_kwargs={"torch_dtype": torch.float16})
    model.max_seq_length = 512

    print(f"Reading {INPUT_FILE}...")
    texts, post_ids = [], []
    import gzip as _gzip
    _open = _gzip.open if INPUT_FILE.endswith(".gz") else open
    with _open(INPUT_FILE, "rt", encoding="utf-8") as f:
        for line in f:
            d = _json.loads(line)
            if "text" in d and "post_id" in d:
                texts.append(d["text"])
                post_ids.append(d["post_id"])
    print(f"  Loaded {len(texts):,} posts")

    print(f"Encoding (batch_size={BATCH_SIZE})... ~2h on T4")
    embeddings = model.encode(texts, batch_size=BATCH_SIZE,
                              show_progress_bar=True, convert_to_numpy=True).astype(np.float32)

    np.save("embeddings.npy", embeddings)
    np.save("post_ids.npy",   np.array(post_ids))
    print(f"\n✓ DS1 done: embeddings.npy {embeddings.shape} → saved to Drive")


---
## DS2 — BERTopic Clustering
~30 min on CPU. Skipped if `post_topics.jsonl` already in Drive. Outputs `topic_info.csv` for browsing clusters.


In [ ]:
# ── DS2: BERTopic Clustering ─────────────────────────────────────────────────
if os.path.exists("post_topics.jsonl") and os.path.exists("topic_info.csv"):
    import pandas as pd
    n_clusters = len(pd.read_csv("topic_info.csv"))
    print(f"✓ Skipping BERTopic — post_topics.jsonl exists ({n_clusters} clusters)")
    print("  Browse topic_info.csv to find cluster IDs for new topics")
else:
    import json as _json, time, numpy as np
    from bertopic import BERTopic
    from bertopic.dimensionality import BaseDimensionalityReduction
    from sklearn.feature_extraction.text import CountVectorizer
    from umap import UMAP
    from hdbscan import HDBSCAN

    SAMPLE_SIZE = 500_000
    t0 = time.time()

    print("Loading texts...")
    texts, pid_list = [], []
    with open("candidate_posts_eng10.jsonl", encoding="utf-8") as f:
        for line in f:
            d = _json.loads(line)
            if "text" in d and "post_id" in d:
                texts.append(d["text"])
                pid_list.append(d["post_id"])
    print(f"  {len(texts):,} texts  ({time.time()-t0:.0f}s)")

    print("Loading embeddings...")
    embeddings = np.load("embeddings.npy", mmap_mode="r")
    post_ids   = np.load("post_ids.npy", allow_pickle=True)
    print(f"  {embeddings.shape}  ({time.time()-t0:.0f}s)")

    print(f"Sampling {SAMPLE_SIZE:,} posts...")
    np.random.seed(42)
    sample_idx        = np.sort(np.random.choice(len(texts), SAMPLE_SIZE, replace=False))
    sample_texts      = [texts[i] for i in sample_idx]
    sample_embeddings = np.array(embeddings[sample_idx], dtype=np.float32)

    print("Running UMAP (~20-35 min)...")
    umap_model     = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                          metric="cosine", random_state=42, low_memory=True, verbose=True)
    reduced_sample = umap_model.fit_transform(sample_embeddings)
    print(f"  UMAP done  ({time.time()-t0:.0f}s)")

    print("Clustering with HDBSCAN + BERTopic...")
    hdbscan_model    = HDBSCAN(min_samples=25, min_cluster_size=250,
                               metric="euclidean", prediction_data=True, core_dist_n_jobs=-1)
    vectorizer_model = CountVectorizer(stop_words="english", min_df=50,
                                       max_df=0.9, max_features=200_000, ngram_range=(1,2))
    topic_model      = BERTopic(umap_model=BaseDimensionalityReduction(),
                                hdbscan_model=hdbscan_model,
                                vectorizer_model=vectorizer_model,
                                calculate_probabilities=False, verbose=True)
    sample_topics, _ = topic_model.fit_transform(sample_texts, reduced_sample)
    print(f"  {len(topic_model.get_topic_info())-1} topics discovered  ({time.time()-t0:.0f}s)")

    all_topics = np.full(len(texts), -1, dtype=np.int32)
    all_topics[sample_idx] = sample_topics

    print("Transforming remaining posts in batches...")
    remaining_idx = [i for i in range(len(texts)) if i not in set(sample_idx)]
    BATCH = 50_000
    for start in range(0, len(remaining_idx), BATCH):
        batch_idx     = remaining_idx[start:start+BATCH]
        batch_emb     = np.array(embeddings[batch_idx], dtype=np.float32)
        batch_reduced = umap_model.transform(batch_emb)
        batch_topics, _ = topic_model.transform([texts[i] for i in batch_idx], batch_reduced)
        for i, t in zip(batch_idx, batch_topics):
            all_topics[i] = t
        print(f"  {min(start+BATCH, len(remaining_idx)):,}/{len(remaining_idx):,}  ({time.time()-t0:.0f}s)")

    print("Saving post_topics.jsonl...")
    with open("post_topics.jsonl", "w") as f:
        for pid, topic in zip(post_ids, all_topics):
            f.write(_json.dumps({"post_id": str(pid), "topic": int(topic)}) + "\n")

    topic_info = topic_model.get_topic_info()
    topic_info.to_csv("topic_info.csv", index=False)
    topic_model.save("bertopic_model", serialization="safetensors", save_ctfidf=True)

    print(f"\n✓ DS2 done in {(time.time()-t0)/60:.1f} min")
    print(f"  post_topics.jsonl — {len(texts):,} rows")
    print(f"  topic_info.csv    — {len(topic_info)} clusters  ← browse this to pick new topics")


---
## DS3 — LR Quality Scoring
Skipped if `candidate_pool.jsonl` already in Drive.


In [ ]:
# ── DS3: Logistic Regression Quality Scoring ─────────────────────────────────
if PIPELINE not in ("dataset", "both"):
    print("Skipping dataset pipeline.")
elif os.path.exists(f"candidate_pool_{TOP_N_PER_TOPIC}.jsonl"):
    print(f"✓ Skipping LR scoring — candidate_pool_{TOP_N_PER_TOPIC}.jsonl exists")
else:
    import csv, json as _json, numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from collections import defaultdict

    cluster_to_topic  = {t["cluster_id"]: t["name"] for t in TOPICS}
    selected_clusters = set(cluster_to_topic.keys())

    if os.path.exists("probability_scores.jsonl"):
        print("Reusing existing probability_scores.jsonl...")
        scores, post_topic = {}, {}
        with open("probability_scores.jsonl") as f:
            for line in f:
                d = _json.loads(line)
                scores[str(d["post_id"])]     = d["probability_score"]
                post_topic[str(d["post_id"])] = d["cluster_id"]
        print(f"  Loaded {len(scores):,} scores")
    else:
        print("Training LR scorer...")
        labeled = {}
        with open(f"{REPO_DIR}/all_labels.csv") as f:
            for row in csv.DictReader(f):
                labeled[str(row["post_id"])] = int(row["label"])

        post_ids_all = np.load("post_ids.npy", allow_pickle=True).astype(str)
        embeddings   = np.load("embeddings.npy", mmap_mode="r")
        pid_to_idx   = {pid: i for i, pid in enumerate(post_ids_all)}

        X_rows, y_rows = [], []
        for pid, label in labeled.items():
            idx = pid_to_idx.get(pid)
            if idx is not None:
                X_rows.append(embeddings[idx]); y_rows.append(label)

        scaler = StandardScaler()
        clf    = LogisticRegression(C=1.0, max_iter=1000, class_weight="balanced", random_state=42)
        clf.fit(scaler.fit_transform(np.array(X_rows, dtype=np.float32)), np.array(y_rows))
        print("  LR trained")

        post_topic = {}
        with open("post_topics.jsonl") as f:
            for line in f:
                d = _json.loads(line)
                if d["topic"] in selected_clusters:
                    post_topic[str(d["post_id"])] = d["topic"]

        eligible = [pid for pid in post_topic if pid in pid_to_idx]
        scores   = {}
        for i in range(0, len(eligible), 50_000):
            batch  = eligible[i:i+50_000]
            idxs   = [pid_to_idx[p] for p in batch]
            probs  = clf.predict_proba(scaler.transform(np.array(embeddings[idxs], dtype=np.float32)))[:, 1]
            for pid, score in zip(batch, probs): scores[pid] = float(score)
            print(f"  Scored {min(i+50_000,len(eligible)):,}/{len(eligible):,}")

        with open("probability_scores.jsonl", "w") as f:
            for pid, score in scores.items():
                f.write(_json.dumps({"post_id":pid,"cluster_id":post_topic[pid],
                                     "probability_score":round(score,4)}) + "\n")

    print("Building candidate pool...")
    post_data = {}
    with open("candidate_posts_eng10.jsonl") as f:
        for line in f:
            d = _json.loads(line); pid = str(d["post_id"])
            if pid in scores: post_data[pid] = d

    by_topic = defaultdict(list)
    for pid, score in scores.items():
        if pid in post_topic:
            by_topic[cluster_to_topic[post_topic[pid]]].append((pid, score))

    total = 0
    with open(f"candidate_pool_{TOP_N_PER_TOPIC}.jsonl", "w") as f:
        for topic_name, pid_scores in sorted(by_topic.items()):
            top = sorted(pid_scores, key=lambda x: x[1], reverse=True)[:TOP_N_PER_TOPIC]
            for pid, score in top:
                d = post_data.get(pid, {})
                f.write(_json.dumps({"post_id":pid,"topic_name":topic_name,
                    "cluster_id":post_topic[pid],"text":d.get("text",""),
                    "engagement":d.get("engagement",0),"lr_score":round(score,4)},
                    ensure_ascii=False) + "\n")
            total += len(top)
            print(f"  {topic_name:<30} {len(top)} posts")
    print(f"\n✓ DS3 done: {total} posts → candidate_pool_{TOP_N_PER_TOPIC}.jsonl")


---
## DS4 — Agent Quality Control
Skipped if final dataset file already in Drive.


In [ ]:
# ── DS4: Agent Quality Control ───────────────────────────────────────────────
if PIPELINE not in ("dataset", "both"):
    pass
elif os.path.exists(OUTPUT_DS):
    print(f"✓ Skipping agent QC — {OUTPUT_DS} exists")
else:
    import json as _json, time
    from anthropic import Anthropic
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from collections import defaultdict

    client = Anthropic()
    SYS = ("You are a content quality rater. Score each post 0.0–1.0.\n"
           "HIGH (0.7-1.0): specific knowledge, insight, concrete facts, in English.\n"
           "LOW (0.0-0.3): personal feelings, list format, not English, low-effort.\n"
           "Reply ONLY with a JSON array of scores. Example: [0.9, 0.4, 0.1]")

    def score_batch(batch, attempt=1):
        numbered = "\n\n".join(f"[{i+1}] {p['text'][:600]}" for i,p in enumerate(batch))
        try:
            resp = client.messages.create(model="claude-haiku-4-5-20251001",
                       max_tokens=200, temperature=0, system=SYS,
                       messages=[{"role":"user","content":f"Score these {len(batch)} posts:\n\n{numbered}"}])
            s = _json.loads(resp.content[0].text.strip())
            if len(s)==len(batch): return [float(x) for x in s]
        except Exception:
            if attempt<3: time.sleep(2**attempt); return score_batch(batch,attempt+1)
        return [0.5]*len(batch)

    posts = [_json.loads(l) for l in open(f"candidate_pool_{TOP_N_PER_TOPIC}.jsonl")]
    print(f"Loaded {len(posts)} posts")
    CKPT = "agent_qc_checkpoint.json"
    cp   = _json.load(open(CKPT)) if os.path.exists(CKPT) else {}
    todo = [p for p in posts if str(p["post_id"]) not in cp]
    print(f"Scoring {len(todo)} posts ({len(cp)} already done)...")

    batches = [todo[i:i+10] for i in range(0,len(todo),10)]
    with ThreadPoolExecutor(max_workers=15) as ex:
        futures = {ex.submit(score_batch,b):b for b in batches}
        done = 0
        for future in as_completed(futures):
            batch = futures[future]
            for post,score in zip(batch,future.result()): cp[str(post["post_id"])]=score
            done += len(batch)
            if done%500==0:
                with open(CKPT,"w") as f: _json.dump(cp,f)
                print(f"  {sum(1 for p in posts if str(p['post_id']) in cp)}/{len(posts)} scored")
    with open(CKPT,"w") as f: _json.dump(cp,f)

    by_topic = defaultdict(list)
    for p in posts: by_topic[p["topic_name"]].append((p,cp.get(str(p["post_id"]),0)))

    total = 0
    with open(OUTPUT_DS,"w") as f:
        for topic,tp in sorted(by_topic.items()):
            top = sorted(tp,key=lambda x:x[1],reverse=True)[:TOP_N_PER_TOPIC]
            for p,score in top: f.write(_json.dumps({**p,"agent_score":score},ensure_ascii=False)+"\n")
            total += len(top)
            print(f"  {topic:<30} {len(top)} posts  cutoff={top[-1][1]:.2f}")
    print(f"\n✓ DS4 done: {total} posts → {OUTPUT_DS}")


---
## DS5 — Label Style Axes
Skipped if all posts already have `axes_json`.


In [ ]:
# ── DS5: Label Style Axes ────────────────────────────────────────────────────
if PIPELINE not in ("dataset", "both"):
    pass
else:
    import json as _json, time
    from anthropic import Anthropic
    from concurrent.futures import ThreadPoolExecutor, as_completed

    posts   = [_json.loads(l) for l in open(OUTPUT_DS)]
    already = sum(1 for p in posts if p.get("axes_json"))
    if already == len(posts):
        print(f"✓ Skipping label axes — all {len(posts)} posts already labeled")
    else:
        keys_str = "\n".join(f'  "{a["name"]}":{{"score":0.0-1.0}},' for a in AXES)
        desc_str = "\n".join(f'- {a["name"]}: {a["label_description"]}' for a in AXES)
        SYS = (f"You are a writing style analyst. Score {len(AXES)} dimensions per post. "
               f"Return ONLY a JSON array.\n\nEach element:\n{{\n{keys_str}\n}}\n\n"
               f"Guidelines:\n{desc_str}\n\nReply ONLY with a JSON array, one object per post.")

        client = Anthropic()

        def score_batch(batch, attempt=1):
            numbered = "\n\n".join(f"[{i+1}] {p['text'][:500]}" for i,p in enumerate(batch))
            try:
                resp = client.messages.create(model="claude-haiku-4-5-20251001",
                           max_tokens=2000, temperature=0, system=SYS,
                           messages=[{"role":"user","content":f"Score these {len(batch)} posts:\n\n{numbered}"}])
                raw = resp.content[0].text.strip()
                if raw.startswith("```"): raw = raw.split("```")[1].lstrip("json")
                r = _json.loads(raw.strip())
                if len(r)==len(batch): return r
            except Exception:
                if attempt<3: time.sleep(2**attempt); return score_batch(batch,attempt+1)
            return [None]*len(batch)

        CKPT = "label_axes_checkpoint.json"
        cp   = _json.load(open(CKPT)) if os.path.exists(CKPT) else {}
        todo = [p for p in posts if str(p["post_id"]) not in cp and not p.get("axes_json")]
        print(f"Labeling {len(todo)} posts ({len(cp)} already done)...")

        batches = [todo[i:i+5] for i in range(0,len(todo),5)]
        with ThreadPoolExecutor(max_workers=20) as ex:
            futures = {ex.submit(score_batch,b):b for b in batches}
            done = 0
            for future in as_completed(futures):
                batch = futures[future]
                for post,axes in zip(batch,future.result()):
                    if axes: cp[str(post["post_id"])]=axes
                done += len(batch)
                if done%100==0:
                    with open(CKPT,"w") as f: _json.dump(cp,f)
                    print(f"  {len(cp)}/{len(posts)} labeled")
        with open(CKPT,"w") as f: _json.dump(cp,f)

        with open(OUTPUT_DS,"w") as f:
            for p in posts:
                p["axes_json"] = cp.get(str(p["post_id"]),p.get("axes_json"))
                f.write(_json.dumps(p,ensure_ascii=False)+"\n")
        print(f"\n✓ DS5 done: {OUTPUT_DS} updated with axes_json")


---
## SAE1 — Generate Rewrites
Skipped if `sae_clean_rewrites.json` already in Drive.


In [ ]:
# ── SAE1: Generate Rewrites ──────────────────────────────────────────────────
if PIPELINE not in ("sae", "both"):
    print("Skipping SAE pipeline.")
elif os.path.exists("sae_clean_rewrites.json"):
    import json as _json
    raw   = _json.load(open("sae_clean_rewrites.json"))
    posts = raw.get("posts", raw) if isinstance(raw,dict) else raw
    print(f"✓ Skipping rewrite generation — sae_clean_rewrites.json exists ({len(posts)} posts)")
else:
    import json as _json, time, pandas as pd
    from anthropic import Anthropic

    axis_names = [a["name"] for a in AXES]
    axis_defs  = {a["name"]:{"up":a["rewrite_up"],"down":a["rewrite_down"]} for a in AXES}

    def make_prompt(text, axis, direction, current, target):
        desc = axis_defs[axis][direction]
        return (f"Rewrite this post so it is {desc}.\n\n"
                f"Current {axis} score: {current:.2f}/1.0\n"
                f"Target  {axis} score: {target:.2f}/1.0\n\n"
                f"RULES:\n- Keep the EXACT same facts, claim, and stance.\n"
                f"- Only change delivery on this one dimension.\n"
                f"- Keep roughly the same length (±20%).\n"
                f"- Return ONLY the rewritten post text.\n\nORIGINAL POST:\n{text}")

    client = Anthropic()

    def gen_rewrite(text, axis, direction, current, target, retries=3):
        for attempt in range(retries):
            try:
                r = client.messages.create(model="claude-haiku-4-5-20251001", max_tokens=400,
                        messages=[{"role":"user","content":make_prompt(text,axis,direction,current,target)}])
                return r.content[0].text.strip()
            except Exception:
                if attempt<retries-1: time.sleep(2**attempt)
        return None

    rows = [_json.loads(l) for l in open(OUTPUT_DS)]
    df   = pd.DataFrame(rows); df["post_id"] = df["post_id"].astype(str)

    def parse_axes(val):
        a = _json.loads(val) if isinstance(val,str) else val
        return {ax:(a[ax]["score"] if isinstance(a.get(ax),dict) else a.get(ax)) for ax in axis_names}

    axes_df = df["axes_json"].apply(parse_axes).apply(pd.Series).astype(float)
    df = pd.concat([df, axes_df], axis=1)

    cache = {}
    CACHE_FILE = "sae_clean_rewrites.json"

    df["mid_count"] = df.apply(lambda row: sum(1 for ax in axis_names if 0.2<=row[ax]<=0.8), axis=1)
    selected_ids = []
    for topic, group in df.groupby("topic_name"):
        selected_ids += group.sort_values("mid_count",ascending=False)["post_id"].iloc[:N_PER_TOPIC_SAE].tolist()
    print(f"Selected {len(selected_ids)} posts ({N_PER_TOPIC_SAE}/topic)")

    todo = []
    for pid in selected_ids:
        row = df[df["post_id"]==pid].iloc[0]
        for ax in axis_names:
            score = float(row[ax]); direction = "up" if score<0.5 else "down"
            target = min(score+0.4,1.0) if direction=="up" else max(score-0.4,0.0)
            if (pid,ax) not in cache:
                todo.append({"post_id":pid,"axis":ax,"direction":direction,
                             "current":score,"target":target,"original":row["text"]})
    print(f"Generating {len(todo)} rewrites (~${len(todo)*0.00062:.2f})...")

    def save_cache():
        by_post = {}
        for (pid,ax),rw in cache.items():
            if pid not in by_post:
                orig = df[df["post_id"]==pid].iloc[0]["text"] if not df[df["post_id"]==pid].empty else ""
                by_post[pid] = {"post_id":pid,"original":orig,"rewrites":{}}
            score = float(df[df["post_id"]==pid].iloc[0][ax]) if not df[df["post_id"]==pid].empty else 0.5
            by_post[pid]["rewrites"][ax] = {"direction":"up" if score<0.5 else "down","rewrite":rw}
        with open(CACHE_FILE,"w") as f:
            _json.dump({"__axis_names__":axis_names,"posts":list(by_post.values())},f,indent=2,ensure_ascii=False)

    failed = 0
    for i,item in enumerate(todo):
        r = gen_rewrite(item["original"],item["axis"],item["direction"],item["current"],item["target"])
        if r: cache[(item["post_id"],item["axis"])]=r
        else: failed+=1
        if (i+1)%50==0:
            save_cache(); print(f"  {i+1}/{len(todo)} done | {failed} failed")
    save_cache()
    print(f"\n✓ SAE1 done: {len(cache)} rewrites → {CACHE_FILE}  ({failed} failed)")


---
## SAE2 — Qwen Embeddings
~30 min on T4 GPU. Skipped if `diff_vectors_*.npy` already in Drive.


In [ ]:
# ── SAE2: Qwen2.5-3B Embeddings (GPU) ───────────────────────────────────────
import glob as _glob
existing_diff = sorted(_glob.glob("diff_vectors_*.npy"))

if PIPELINE not in ("sae", "both"):
    pass
elif existing_diff:
    print(f"✓ Skipping Qwen embedding — {existing_diff[-1]} exists")
elif not os.path.exists("sae_clean_rewrites.json"):
    print("⚠ sae_clean_rewrites.json not found. Run SAE1 first.")
else:
    import json as _json, numpy as np, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    if not torch.cuda.is_available():
        raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

    raw        = _json.load(open("sae_clean_rewrites.json"))
    axis_names = raw.get("__axis_names__",[a["name"] for a in AXES])
    posts_list = raw.get("posts",raw) if isinstance(raw,dict) else raw
    posts_list = [p for p in posts_list if len(p.get("rewrites",{}))==len(axis_names)]
    records    = [{"post_id":p["post_id"],"axis":ax,"direction":p["rewrites"][ax]["direction"],
                   "original":p["original"],"rewrite":p["rewrites"][ax]["rewrite"]}
                  for p in posts_list for ax in axis_names]
    print(f"Posts: {len(posts_list)}  Pairs: {len(records)}")

    MODEL_NAME = "Qwen/Qwen2.5-3B"
    tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype=torch.bfloat16,low_cpu_mem_usage=True).to("cuda")
    model.eval()
    print(f"Model loaded")

    captured = {}
    def hook(module,inputs,output): captured["acts"]=output[0] if isinstance(output,tuple) else output
    handle = model.model.layers[23].register_forward_hook(hook)

    def embed_texts(texts):
        vecs = []
        for i in range(0,len(texts),2):
            batch = texts[i:i+2]
            enc   = tokenizer(batch,padding=True,truncation=True,max_length=512,return_tensors="pt").to("cuda")
            with torch.no_grad(): model(**enc,use_cache=False)
            acts   = captured["acts"].float()
            mask   = enc.attention_mask.unsqueeze(-1).float()
            pooled = (acts*mask).sum(1)/mask.sum(1).clamp(min=1)
            vecs.append(pooled.cpu().numpy())
            if (i//2+1)%200==0: print(f"  {i+len(batch)}/{len(texts)}")
        return np.vstack(vecs).astype(np.float32)

    print("Embedding originals...")
    emb_orig = embed_texts([r["original"] for r in records])
    print("Embedding rewrites...")
    emb_rew  = embed_texts([r["rewrite"]  for r in records])
    handle.remove()

    diff = emb_rew - emb_orig; n = len(records)
    np.save(f"diff_vectors_{n}.npy", diff)
    np.save(f"axis_labels_{n}.npy",  np.array([r["axis"]      for r in records]))
    np.save(f"directions_{n}.npy",   np.array([r["direction"] for r in records]))
    np.save(f"post_ids_{n}.npy",     np.array([r["post_id"]  for r in records]))
    print(f"\n✓ SAE2 done: diff_vectors_{n}.npy  shape={diff.shape}")


---
## SAE3 — Train SAE
Skipped if `sae_qwen_model.pt` already in Drive.


In [ ]:
# ── SAE3: Train Sparse Autoencoder ───────────────────────────────────────────
import glob as _glob
diff_files = sorted(_glob.glob("diff_vectors_*.npy"))

if PIPELINE not in ("sae", "both"):
    pass
elif not diff_files:
    print("⚠ No diff_vectors_*.npy found. Run SAE2 first.")
elif os.path.exists("sae_qwen_model.pt"):
    print("✓ Skipping SAE training — sae_qwen_model.pt exists")
else:
    import numpy as np, torch, torch.nn as nn
    from collections import Counter

    N_FEATURES=64; L1_COEF=0.05; LR_SAE=1e-3; EPOCHS=600; BATCH=128
    CONFIRM_LIFT=0.30; PARTIAL_LIFT=0.15

    diff_file = diff_files[-1]; n_str = diff_file.split("_")[-1].replace(".npy","")
    diff_vecs   = np.load(diff_file).astype(np.float32)
    axis_labels = np.load(f"axis_labels_{n_str}.npy")
    directions  = np.load(f"directions_{n_str}.npy")
    input_dim   = diff_vecs.shape[1]
    axis_names_local = [a["name"] for a in AXES]
    print(f"Loaded {diff_file}  shape={diff_vecs.shape}")

    diff_vecs[directions=="down"] *= -1
    diff_vecs = diff_vecs / np.linalg.norm(diff_vecs,axis=1,keepdims=True).clip(min=1e-8)

    class SAE(nn.Module):
        def __init__(self,dim,n,seed=42):
            super().__init__()
            g=torch.Generator().manual_seed(seed)
            w=torch.empty(n,dim); nn.init.kaiming_uniform_(w,a=5**0.5,generator=g)
            self.W_enc=nn.Parameter(w); self.b_enc=nn.Parameter(torch.zeros(n))
            wd=torch.randn(dim,n,generator=g); wd=wd/wd.norm(dim=0,keepdim=True).clamp_min(1e-8)
            self.W_dec=nn.Parameter(wd); self.b_dec=nn.Parameter(torch.zeros(dim))
        def encode(self,x): return torch.relu(x@self.W_enc.T+self.b_enc)
        def forward(self,x): f=self.encode(x); return f,f@self.W_dec.T+self.b_dec
        @torch.no_grad()
        def normalize_decoder(self): self.W_dec.data.div_(self.W_dec.norm(dim=0,keepdim=True).clamp_min(1e-8))

    torch.manual_seed(42)
    model=SAE(input_dim,N_FEATURES); optimizer=torch.optim.Adam(model.parameters(),lr=LR_SAE)
    X=torch.from_numpy(diff_vecs)
    print(f"Training SAE (F={N_FEATURES}, L1={L1_COEF}, epochs={EPOCHS})...")

    for epoch in range(EPOCHS):
        perm=torch.randperm(len(X))
        for i in range(0,len(X),BATCH):
            b=X[perm[i:i+BATCH]]; optimizer.zero_grad()
            feats,recon=model(b)
            loss=((b-recon)**2).sum(1).mean()+L1_COEF*feats.abs().sum(1).mean()
            loss.backward(); optimizer.step(); model.normalize_decoder()
        if (epoch+1)%100==0:
            with torch.no_grad(): dens=(model.encode(X)>0).float().mean().item()
            print(f"  Epoch {epoch+1}/{EPOCHS}  density={dens:.3f}")

    torch.save(model.state_dict(),"sae_qwen_model.pt")

    with torch.no_grad(): acts=model.encode(X).numpy()
    axis_arr=np.array(axis_labels); cats=Counter()
    print("\nPer-axis coverage:")
    for ax in axis_names_local:
        best_lift,best_f=0.0,-1
        for f in range(N_FEATURES):
            feat=acts[:,f]
            if (feat>0).mean()<0.01: continue
            mask=axis_arr==ax
            lift=float((feat[mask]>0).mean())-float((feat[~mask]>0).mean())
            if abs(lift)>abs(best_lift): best_lift,best_f=lift,f
        s="✓" if abs(best_lift)>=CONFIRM_LIFT else "~" if abs(best_lift)>=PARTIAL_LIFT else "✗"
        print(f"  {s} {ax:<25} best=F{best_f}  lift={best_lift:.3f}")
        cats["confirms_axis" if abs(best_lift)>=CONFIRM_LIFT else "partial_overlap" if abs(best_lift)>=PARTIAL_LIFT else "novel_candidate"]+=1
    print(f"\n✓ SAE3 done: {dict(cats)}  →  sae_qwen_model.pt")


---
## SAE4 — Feature Labeler
Generates interactive HTML viewer. Downloads automatically.


In [ ]:
# ── SAE4: Generate Feature Labeler HTML ──────────────────────────────────────
if PIPELINE not in ("sae", "both"):
    pass
elif not os.path.exists("sae_qwen_model.pt"):
    print("⚠ sae_qwen_model.pt not found. Run SAE3 first.")
else:
    import subprocess
    result = subprocess.run(["python", f"{REPO_DIR}/sae4_labeler.py"],
                            cwd=WORK_DIR, capture_output=True, text=True)
    if result.returncode != 0: print("STDERR:", result.stderr)
    else:
        from google.colab import files
        files.download("sae_feature_labeler.html")
        print("✓ SAE4 done — sae_feature_labeler.html downloaded. Open in browser to explore features.")


---
## Output


In [ ]:
# ── Summary + Downloads ───────────────────────────────────────────────────────
import os, glob as _glob, json as _json
from google.colab import files

print("=" * 55)
print("  PIPELINE COMPLETE")
print("=" * 55)
print(f"  All files saved to: {WORK_DIR}")

if PIPELINE in ("dataset","both") and os.path.exists(OUTPUT_DS):
    posts   = [_json.loads(l) for l in open(OUTPUT_DS)]
    labeled = sum(1 for p in posts if p.get("axes_json"))
    print(f"\n Dataset  →  {OUTPUT_DS}")
    print(f"  {len(posts)} posts  ·  {len(TOPICS)} topics  ·  {labeled} labeled with axes")
    files.download(OUTPUT_DS)

if PIPELINE in ("sae","both"):
    dv = sorted(_glob.glob("diff_vectors_*.npy"))
    if dv:
        import numpy as np
        print(f"\n SAE diff vectors  →  {dv[-1]}  shape={np.load(dv[-1],mmap_mode='r').shape}")
    if os.path.exists("sae_qwen_model.pt"):
        print(f"  SAE model  →  sae_qwen_model.pt")
    if os.path.exists("sae_feature_labeler.html"):
        print(f"  Feature viewer  →  sae_feature_labeler.html  (open in browser)")

if os.path.exists("topic_info.csv"):
    print(f"\n topic_info.csv  →  browse to find cluster IDs for new topics")
